In [ ]:
%pip install -qU --force-reinstall "numpy<2.1" "pillow<12.0" scikit-learn==1.6 matplotlib seaborn phik category_encoders optuna catboost xgboost lightgbm pandas==2.2.2

In [ ]:
import sys

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import phik

import optuna

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate, cross_val_predict
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score, precision_recall_curve, make_scorer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, StackingClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from IPython.display import display

pd.set_option('display.max_columns', None)

In [ ]:
RANDOM_STATE=42

In [ ]:
def get_df(path):
    return pd.read_csv(path, sep=',', decimal='.')

In [ ]:
train = get_df('./datasets/train.csv')

In [ ]:
display(train.head())
print()
train.info()

In [ ]:
int_cols = train.select_dtypes(include=['int64'])
float_cols = train.select_dtypes(include=['float64'])

for col in int_cols:
    train[col] = pd.to_numeric(train[col], downcast='integer')

for col in float_cols:
    train[col] = pd.to_numeric(train[col], downcast='float')

train.info()

In [ ]:
print('Явные дубликаты:', train.duplicated().sum())
train = train.drop_duplicates()

In [ ]:
train = train.drop(columns=['id'])

In [ ]:
eda_df = train.copy()

In [ ]:
target = 'Heart Disease'

y_desc = train[target].describe()
y_desc

In [ ]:
y_counts = train[target].value_counts()

def show_bar(data, title):
    ax = data.plot(
        kind='bar',
        title=title,
        ylabel='Кол-во',
        rot=0,
        figsize=(12, 4)
    )

    ax.bar_label(ax.containers[0], label_type='center')
    ax.grid(axis='y', linestyle='--', alpha=.7)

    plt.show()
    plt.close()

show_bar(y_counts, 'Распределение целевой переменной')

In [ ]:
print(f'Доля Absence: {y_counts['Absence'] / y_desc['count'] * 100:.0f}%')
print(f'Доля Presence: {y_counts['Presence'] / y_desc['count'] * 100:.0f}%')

In [ ]:
data = []

num_cols = []
cat_cols = []
for col in eda_df.columns:
    if col == target:
        continue

    nu = eda_df[col].nunique()
    data.append({
        'Признак': col,
        'Кол-во': nu
    })

    if nu > 5:
        num_cols.append(col)
    else:
        cat_cols.append(col)

pd.DataFrame(data).sort_values('Кол-во', ascending=False)

In [ ]:
def show_boxplot(col, data, ax):
    sns.boxplot(data=data, ax=ax)

    ax.set_title(f'Размах значений признака: {col}')
    ax.set_ylabel('Размах')
    ax.grid(True)

def show_hist(col, data, ax):
    data.plot(
        kind='hist',
        bins=50,
        grid=True,
        ax=ax
    )

    ax.set_title(f'Распределение значений признака: {col}')
    ax.set_ylabel('Кол-во значений')
    ax.set_yscale('log')

def show_hist(col, data, ax):
    data.plot(
        kind='hist',
        bins=50,
        grid=True,
        ax=ax
    )

    ax.set_title(f'Распределение значений признака: {col}')
    ax.set_ylabel('Кол-во значений')
    ax.set_yscale('log')

for col in num_cols:
    _, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 4), constrained_layout=True)

    data = eda_df[col]
    show_hist(col, data, axes[0])
    show_boxplot(col, data, axes[1])

    plt.show()
    plt.close()


In [ ]:
for col in cat_cols:
    show_bar(eda_df[col].value_counts(), f'Распределение признака: {col}')

In [ ]:
def show_heatmap(corr, title, x, y_coef):
    num_rows = len(corr)
    dynamic_height = max(5, num_rows * y_coef)
    plt.figure(figsize=(x, dynamic_height))

    sns.heatmap(
        corr,
        annot=True,
        fmt='.2f',
        cmap='coolwarm',
        linewidth=.5,
        cbar=False
    )

    plt.title(title)
    plt.show()

In [ ]:
nominal_cols = ['Chest pain type', 'EKG results', 'Thallium', 'Slope of ST']

In [ ]:
mapping = {
    'Chest pain type': {
        1: 'Typical Angina', 2: 'Atypical Angina',
        3: 'Non-anginal Pain', 4: 'Asymptomatic'
    },
    'EKG results': {
        0: 'Normal', 1: 'ST-T Abnormality',
        2: 'LV Hypertrophy'
    },
    'Slope of ST': {
        1: 'Upsloping', 2: 'Flat', 3: 'Downsloping'
    },
    'Thallium': {
        3: 'Normal', 6: 'Fixed Defect', 7: 'Reversible Defect'
    }
}

for col, labels in mapping.items():
    if col in nominal_cols:
        eda_df[col] = eda_df[col].map(labels)

In [ ]:
for col in nominal_cols:
    show_bar(eda_df[col].value_counts(), f'Распределение признака: {col}')

In [ ]:
corr_matrix = eda_df.phik_matrix(interval_cols=['Age', 'Sex', 'BP', 'Cholesterol', 'FBS over 120', 'Max HR', 'Exercise angina', 'ST depression', 'Number of vessels fluro'])
target_corr = corr_matrix[corr_matrix.index != 'Heart Disease'][['Heart Disease']].sort_values('Heart Disease', ascending=False)

show_heatmap(target_corr, title='Корреляция с целевой переменной', x=3, y_coef=.1)

In [ ]:
show_heatmap(corr_matrix, title='Матрица корреляций', x=10, y_coef=.3)

In [ ]:
X = train.drop(columns=['Heart Disease'])
y = train['Heart Disease'].map({'Absence': 0, 'Presence': 1})

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    shuffle=True,
    test_size=.2,
    random_state=RANDOM_STATE
)

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")
print(f"Среднее значенее целевой переменной в train: {y_train.mean():.3f}")
print(f"Среднее значенее целевой переменной в test: {y_test.mean():.3f}")

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
results = []

In [ ]:
def plot_classification_results(y_test, y_prob, y_pred, thresh=0.5, name="Model"):
    _, (ax_conf, ax_prob) = plt.subplots(1, 2, figsize=(16, 6))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax_conf)
    ax_conf.set(
        title=f"{name} - Confusion Matrix",
        xlabel="Предсказанные метки",
        ylabel="Актуальные метки"
    )

    df_probs = pd.DataFrame({'Probability': y_prob, 'Actual': y_test})

    sns.histplot(data=df_probs, x='Probability', hue='Actual',
                 element='step', kde=True, bins=30, ax=ax_prob, palette='magma')

    ax_prob.axvline(thresh, color="red", linestyle="--", label=f'Порог {thresh:.2f}')
    ax_prob.set(
        title=f"{name} - Вероятностное распределение по классам",
        xlabel="Предполагаемая вероятность (1)",
        ylabel="Кол-во",
    )
    ax_prob.legend(loc='upper left')

    plt.tight_layout()
    plt.show()

def show_ROC_curve(y_true, y_proba, score, name):
    fpr, tpr, _ = roc_curve(y_true, y_proba)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC-кривая (AUC = {score:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate (FPR)')
    plt.ylabel('True Positive Rate (TPR)')
    plt.title(f'ROC-кривая: {name}')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.show()

def evaluate_model(pipeline, X_train, y_train, name="Model"):
    y_proba = cross_val_predict(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]
    score = roc_auc_score(y_train, y_proba)

    print(f"[{name}] ROC-AUC: {score:.4f}\n")
    show_ROC_curve(y_train, y_proba, score, name)

    y_pred = (y_proba >= .5).astype(int)
    print("Результаты на тесте с новым порогом")
    print(classification_report(y_train, y_pred, zero_division=0))

    plot_classification_results(y_train, y_proba, y_pred, name=name)

    results.append({
        'Название модели': name,
        'ROC-AUC': score
    })

In [ ]:
def run_optuna(build_pipeline_fn, build_best_pipeline_fn, name, n_trials=30):
    def objective(trial):
        pipeline = build_pipeline_fn(trial)

        scores = cross_val_score(
            pipeline,
            X_train,
            y_train,
            cv=cv,
            scoring="roc_auc",
            n_jobs=-1
        )

        mean_score = scores.mean()

        trial.report(mean_score, step=0)
        if trial.should_prune():
            raise optuna.TrialPruned()

        return mean_score

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction='maximize', sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True, n_jobs=-1)

    fig1 = optuna.visualization.plot_optimization_history(study)
    fig2 = optuna.visualization.plot_param_importances(study)

    display(fig1)
    display(fig2)

    best_params = study.best_params
    print("Лучшие гиперпараметры:", best_params)
    best_value = study.best_value
    print("Лучшее среднее значение ROC-AUC на кросс-валидации:", round(best_value, 3))

    evaluate_model(build_best_pipeline_fn(best_params), X_train, y_train, name=name)

    return best_params

In [ ]:
base_linear_pipeline = Pipeline(steps=[
    ('preprocessor', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))
])

In [ ]:
evaluate_model(base_linear_pipeline, X_train, y_train, name='LogisticRegression [Baseline]')

In [ ]:
upd_num_cols = [col for col in num_cols if col not in cat_cols]

In [ ]:
def medical_mapping_transformer(X):
    X_copy = X.copy()

    for col, labels in mapping.items():
        if col in X_copy.columns:
            X_copy[col] = X_copy[col].map(labels)

    return X_copy

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', make_pipeline(
            FunctionTransformer(medical_mapping_transformer, feature_names_out="one-to-one"),
            OneHotEncoder(drop='first', sparse_output=False)
        ), nominal_cols),
        ('num', StandardScaler(), upd_num_cols)
    ]
).set_output(transform="pandas")

X_transformed_df = preprocessor.fit_transform(X_train)
X_transformed_df.head()

In [ ]:
upd_linear_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))
])

evaluate_model(upd_linear_pipeline, X_train, y_train, name='LogisticRegression [Baseline with upd pipe]')

In [ ]:
def build_pipeline_fn(trial):
    params = {
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        "fit_intercept": True,
        "tol": trial.suggest_float("tol", 1e-4, 1e-2, log=True),
        "max_iter": trial.suggest_int("max_iter", 2000, 20000),
        "solver": trial.suggest_categorical("solver", ["lbfgs", "newton-cg", "newton-cholesky", "saga"]),
        "C": trial.suggest_float("C", 1e-4, 1e2, log=True),
        "random_state": RANDOM_STATE,
    }

    solver = params["solver"]
    if solver in ["lbfgs", "newton-cg", "newton-cholesky"]:
        params["l1_ratio"] = 0
    elif solver == "saga":
        params["l1_ratio"] = trial.suggest_float("l1_ratio", 0.0, 1.0)

    model = LogisticRegression(**params)
    pipeline = Pipeline(steps=[
        ('preprocessor', StandardScaler()),
        ('model', model)
    ])

    return pipeline

def build_best_pipeline_fn(best_params):
    pipeline = Pipeline(steps=[
        ("preprocessor", StandardScaler()),
        ("model", LogisticRegression(**best_params, random_state=RANDOM_STATE, n_jobs=-1))
    ])
    return pipeline

lr_best_params = run_optuna(build_pipeline_fn, build_best_pipeline_fn, "LogisticRegression [Best Params]", n_trials=100)

In [ ]:
def build_pipeline_fn(trial):
    params = {
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        "fit_intercept": True,
        "tol": trial.suggest_float("tol", 1e-4, 1e-2, log=True),
        "max_iter": trial.suggest_int("max_iter", 2000, 20000),
        "solver": trial.suggest_categorical("solver", ["lbfgs", "newton-cg", "newton-cholesky", "saga"]),
        "C": trial.suggest_float("C", 1e-4, 1e2, log=True),
        "random_state": RANDOM_STATE,
    }

    solver = params["solver"]
    if solver in ["lbfgs", "newton-cg", "newton-cholesky"]:
        params["l1_ratio"] = 0
    elif solver == "saga":
        params["l1_ratio"] = trial.suggest_float("l1_ratio", 0.0, 1.0)

    model = LogisticRegression(**params)
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    return pipeline

def build_best_pipeline_fn(best_params):
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(**best_params, random_state=RANDOM_STATE, n_jobs=-1))
    ])
    return pipeline

lr_up_best_params = run_optuna(build_pipeline_fn, build_best_pipeline_fn, "LogisticRegression [Best Params with upd pipe]", n_trials=100)

In [ ]:
evaluate_model(DecisionTreeClassifier(class_weight='balanced'), X_train, y_train, name='Tree [Baseline]')

In [ ]:
def build_pipeline_fn(trial):
    params = {
        "criterion": trial.suggest_categorical("criterion", ["entropy", "log_loss"]),
        "splitter": "best",
        "max_depth": trial.suggest_int("max_depth", 4, 15),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 60),
        "min_samples_split": trial.suggest_int("min_samples_split", 20, 150),
        "max_features": None,
        "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 1e-5, 2e-3, log=True),
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-5, 5e-3, log=True),
        "class_weight": "balanced",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    }

    return DecisionTreeClassifier(**params)

def build_best_pipeline_fn(best_params):
    return DecisionTreeClassifier(**best_params)

tree_best_params = run_optuna(build_pipeline_fn, build_best_pipeline_fn, "Tree [Best Params]", n_trials=50)

In [ ]:
def build_pipeline_fn(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 30, 100),
        "max_depth": trial.suggest_int("max_depth", 5, 15),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 50, 200),
        "min_samples_split": trial.suggest_int("min_samples_split", 50, 300),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3]),
        "bootstrap": True,
        "max_samples": trial.suggest_float("max_samples", 0.3, 0.5),
        "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 1e-6, 1e-2, log=True),
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-6, 1e-2, log=True),
        "criterion": "gini",
        "class_weight": "balanced",
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }

    return RandomForestClassifier(**params)

def build_best_pipeline_fn(best_params):
    return RandomForestClassifier(**best_params)

rf_best_params = run_optuna(build_pipeline_fn, build_best_pipeline_fn, "Random Forest [Best Params]", n_trials=50)

In [ ]:
evaluate_model(CatBoostClassifier(auto_class_weights='Balanced', thread_count=-1, random_state=RANDOM_STATE), X_train, y_train, name='Cat Boost [Baseline]')

In [ ]:
def build_pipeline_fn(trial):
    bootstrap_type = trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"])
    grow_policy = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"])

    params = {
        "iterations": trial.suggest_int("iterations", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 12),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-8, 100.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-8, 10.0, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 100),
        "border_count": trial.suggest_int("border_count", 32, 128),
        "bootstrap_type": bootstrap_type,
        "grow_policy": grow_policy,
        "auto_class_weights": "Balanced",
        "verbose": False,
        "random_state": RANDOM_STATE,
        "task_type": "CPU",
        "thread_count": -1,
    }

    if bootstrap_type == "Bayesian":
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10)
    elif bootstrap_type in ["Bernoulli", "MVS"]:
        params["subsample"] = trial.suggest_float("subsample", 0.1, 1.0)

    if grow_policy == "Lossguide":
        params["max_leaves"] = trial.suggest_int("max_leaves", 16, 64)

    return CatBoostClassifier(**params)

def build_best_pipeline_fn(best_params):
    return CatBoostClassifier(**best_params)

cb_best_params = run_optuna(build_pipeline_fn, build_best_pipeline_fn, "Cat Boost [Best Params]")

In [ ]:
evaluate_model(XGBClassifier(random_state=RANDOM_STATE, n_jobs=-1), X_train, y_train, name='XG Boost [Baseline]')

In [ ]:
def build_pipeline_fn(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 10.0),
        "booster": "gbtree",
        "tree_method": "auto",
        "objective": "binary:logistic",
        "random_state": RANDOM_STATE,
        "verbosity": 0,
        "n_jobs": -1
    }

    return XGBClassifier(**params)

def build_best_pipeline_fn(best_params):
    return XGBClassifier(**best_params)

xgb_best_params = run_optuna(build_pipeline_fn, build_best_pipeline_fn, "XG Boost [Best Params]", n_trials=50)

In [ ]:
evaluate_model(LGBMClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1), X_train, y_train, name='Light Boost [Baseline]')

In [ ]:
def build_pipeline_fn(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 255),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 100),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0, 15),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "boosting_type": "gbdt",
        "class_weight": "balanced",
        "random_state": RANDOM_STATE,
        "verbose": -1,
        "n_jobs": -1,
    }

    return LGBMClassifier(**params)

def build_best_pipeline_fn(best_params):
    return LGBMClassifier(**best_params)

lb_best_params = run_optuna(build_pipeline_fn, build_best_pipeline_fn, "Light Boost [Best Params]", n_trials=50)

In [ ]:
lb_best_model = LGBMClassifier(**lb_best_params)
lr_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(**lr_best_params)
)
cb_best_model = CatBoostClassifier(**cb_best_params)
xgb_best_model = XGBClassifier(**xgb_best_params)
bagging_pipeline = RandomForestClassifier(**rf_best_params)

In [ ]:
def run_stack(estimators, stack_name):
    meta_features = pd.DataFrame()

    print("Генерация признаков для стекинга...")

    for name, model in estimators.items():
        print(f"Обработка {name}...")
        probs = cross_val_predict(
            model,
            X_train,
            y_train,
            cv=cv,
            method='predict_proba',
            n_jobs=-1
        )[:, 1]

        meta_features[f'{name}_pred'] = probs

    print("\nMeta-features head:")
    print(meta_features.head())

    params_meta = {
        'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
        'penalty': ['l1', 'l2'],
        'class_weight': 'balanced',
        'solver': 'liblinear',
    }

    grid_meta = GridSearchCV(
        LogisticRegression(random_state=RANDOM_STATE),
        params_meta,
        cv=5,
        scoring='roc_auc',
        n_jobs=-1
    )
    grid_meta.fit(meta_features, y_train)

    print(f"Лучшие параметры: {grid_meta.best_params_}")
    print(f"Лучший ROC-AUC после тюнинга: {grid_meta.best_score_:.5f}")

    evaluate_model(grid_meta.best_estimator_, meta_features, y_train, name=stack_name)

    plt.figure(figsize=(10, 8))
    sns.heatmap(meta_features.corr(), annot=True, cmap='coolwarm', fmt=".3f")
    plt.title("Корреляция предсказаний базовых моделей")
    plt.show()

    return grid_meta.best_params_

In [ ]:
estimators = {
    'log_reg': lr_pipeline,
    'bagging': bagging_pipeline
}

swb_params = run_stack(estimators, stack_name='Stack without boost')

In [ ]:
estimators = {
    'light_boost': lb_best_model,
    'cat_boost': cb_best_model,
    'xg_boost': xgb_best_model,
}

sb_params = run_stack(estimators, stack_name='Stack Boost')

In [ ]:
estimators = {
    'light_boost': lb_best_model,
    'log_reg': lr_pipeline,
}

top2_params = run_stack(estimators, stack_name='Stack TOP-2')

In [ ]:
estimators = {
    'light_boost': lb_best_model,
    'log_reg': lr_pipeline,
    'cat_boost': cb_best_model,
}

top3_params = run_stack(estimators, stack_name='Stack TOP-3')

In [ ]:
estimators = {
    'light_boost': lb_best_model,
    'log_reg': lr_pipeline,
    'cat_boost': cb_best_model,
    'xg_boost': xgb_best_model,
    'bagging': bagging_pipeline
}

top5_params = run_stack(estimators, stack_name='Stack TOP-5')

In [ ]:
estimators = {
    'light_boost': lb_best_model,
    'log_reg': lr_pipeline,
    'xg_boost': xgb_best_model,
}

corr_top3_params = run_stack(estimators, stack_name='Stack TOP-3 corr')

In [ ]:
pd.DataFrame(results) \
    .sort_values(by='ROC-AUC', ascending=False) \
    .style.background_gradient(cmap='coolwarm')

In [ ]:
estimators = [
    ('light_boost', lb_best_model),
    ('log_reg', lr_pipeline),
]

top2_final_stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(**top2_params, random_state=RANDOM_STATE),
    cv=cv,
    n_jobs=-1
)

print("Финальное тестирование стекинга...")
top2_final_stack.fit(X_train, y_train)

print("Предсказание на тесте...")
y_proba = top2_final_stack.predict_proba(X_test)[:, 1]

score = roc_auc_score(y_test, y_proba)
name = '[Test Stack TOP-3]'
print(f"{name} ROC-AUC: {score:.4f}\n")
show_ROC_curve(y_test, y_proba, score, name)

In [ ]:
estimators = [
    ('light_boost', lb_best_model),
    ('cat_boost', cb_best_model),
    ('xg_boost', xgb_best_model),
]

boost_final_stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(**sb_params, random_state=RANDOM_STATE),
    cv=cv,
    n_jobs=-1
)

print("Финальное тестирование стекинга...")
boost_final_stack.fit(X_train, y_train)

print("Предсказание на тесте...")
y_proba = boost_final_stack.predict_proba(X_test)[:, 1]

score = roc_auc_score(y_test, y_proba)
name = '[Test Stack TOP-5]'
print(f"{name} ROC-AUC: {score:.4f}\n")
show_ROC_curve(y_test, y_proba, score, name)

In [ ]:
estimators = [
    ('light_boost', lb_best_model),
    ('log_reg', lr_pipeline),
    ('xg_boost', xgb_best_model)
]

top3_corr_final_stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(**corr_top3_params, random_state=RANDOM_STATE),
    cv=cv,
    n_jobs=-1
)

print("Финальное тестирование стекинга...")
top3_corr_final_stack.fit(X_train, y_train)

print("Предсказание на тесте...")
y_proba = top3_corr_final_stack.predict_proba(X_test)[:, 1]

score = roc_auc_score(y_test, y_proba)
name = '[Test Stack TOP-3 corr]'
print(f"{name} ROC-AUC: {score:.4f}\n")
show_ROC_curve(y_test, y_proba, score, name)

In [ ]:
test = get_df('./datasets/test.csv')

X_kaggle = test.drop(columns=['id'])

In [ ]:
from google.colab import files

top2_final_stack.fit(X, y)
top2_submission_proba = top3_corr_final_stack.predict_proba(X_kaggle)[:, 1]
submission = pd.DataFrame({
    'id': test['id'],
    'Heart Disease': top2_submission_proba
})
submission.to_csv('datasets/submission1.csv', index=False)
files.download('datasets/submission1.csv')

boost_final_stack.fit(X, y)
boost_submission_proba = top3_corr_final_stack.predict_proba(X_kaggle)[:, 1]
submission = pd.DataFrame({
    'id': test['id'],
    'Heart Disease': boost_submission_proba
})
submission.to_csv('datasets/submission2.csv', index=False)
files.download('datasets/submission2.csv')

top3_corr_final_stack.fit(X, y)
top3_submission_proba = top3_corr_final_stack.predict_proba(X_kaggle)[:, 1]
submission = pd.DataFrame({
    'id': test['id'],
    'Heart Disease': top3_submission_proba
})
submission.to_csv('datasets/submission3.csv', index=False)
files.download('datasets/submission3.csv')